In [1]:
!pip install yfinance dash plotly pandas
import yfinance as yf
import pandas as pd
import plotly.express as px
from dash import Dash, html, dcc, dash_table, Input, Output

tickers = ["PFE", "AZN", "JNJ"]

company_labels = {
    "PFE": "Pfizer",
    "AZN": "AstraZeneca",
    "JNJ": "Johnson & Johnson"
}

company_colors = {
    "PFE": "#21508A",   
    "AZN": "#E3B727",   
    "JNJ": "#E01414"    
}

graph_background = "#E5ECF6"

# DOWNLOAD STOCK DATA FROM YAHOO FINANCE
stock_raw = yf.download(
    tickers,
    start="2018-01-01",
    end="2025-01-01",
    auto_adjust=True
)

close = stock_raw["Close"].reset_index()

stock_data = close.melt(
    id_vars="Date",
    var_name="Company",
    value_name="Close"
)

# Add month
stock_data["Month"] = stock_data["Date"].dt.to_period("M").astype(str)

# Daily return
stock_data["Return"] = stock_data.groupby("Company")["Close"].pct_change()
stock_data["Return (%)"] = stock_data["Return"] * 100

# Moving averages
stock_data["MA50"] = stock_data.groupby("Company")["Close"].transform(
    lambda x: x.rolling(50).mean()
)

stock_data["MA200"] = stock_data.groupby("Company")["Close"].transform(
    lambda x: x.rolling(200).mean()
)

stock_data = stock_data.dropna(subset=["Return"])

# DOWNLOAD FINANCIAL DATA FROM YAHOO FINANCE
financial_list = []

for ticker in tickers:
    data = yf.Ticker(ticker).financials.T.reset_index()
    data = data.rename(columns={"index": "Date"})
    data["Year"] = pd.to_datetime(data["Date"]).dt.year

    data = data[
        (data["Year"] >= 2018) &
        (data["Year"] <= 2024)
    ]

    temp = pd.DataFrame({
        "Year": data["Year"],
        "Company": ticker,
        "Revenue": data["Total Revenue"] / 1_000_000,
        "Net Profit": data["Net Income"] / 1_000_000
    })

    financial_list.append(temp)

financial_data = pd.concat(financial_list, ignore_index=True)
financial_data = financial_data.dropna()
financial_data = financial_data.sort_values(["Company", "Year"])

app = Dash(__name__)

app.layout = html.Div([

    # HEADER
    html.Div([

        html.H1(
            "COVID-19 Impact on Global Pharmaceutical Companies Dashboard",
            style={
                "textAlign": "center",
                "color": "#1F2D3D",
                "fontWeight": "bold",
                "marginBottom": "5px"
            }
        ),

        html.H3(
            "Financial Performance and Stock Market Behaviour: Pfizer, AstraZeneca and Johnson & Johnson",
            style={
                "textAlign": "center",
                "color": "#4A4A4A",
                "fontWeight": "normal",
                "marginTop": "0px"
            }
        ),

        html.P(
            "This dashboard presents a comparative analysis of revenue, net profit and stock market behaviour among selected global pharmaceutical companies before, during and after the COVID-19 period.",
            style={
                "textAlign": "center",
                "fontSize": "16px",
                "color": "#555555"
            }
        )

    ], style={
        "backgroundColor": "#F4F6F8",
        "padding": "25px",
        "borderRadius": "15px",
        "marginBottom": "25px",
        "boxShadow": "0 4px 10px rgba(0,0,0,0.08)"
    }),

    # DROPDOWN
    html.Div([

        html.Label(
            "Select Company:",
            style={
                "fontWeight": "bold",
                "fontSize": "16px",
                "marginBottom": "8px",
                "display": "block"
            }
        ),

        dcc.Dropdown(
            id="company_filter",
            options=[
                {"label": "All Companies", "value": "All"},
                {"label": "Pfizer", "value": "PFE"},
                {"label": "AstraZeneca", "value": "AZN"},
                {"label": "Johnson & Johnson", "value": "JNJ"}
            ],
            value="All",
            clearable=False,
            style={
                "width": "55%",
                "marginBottom": "25px"
            }
        )

    ]),

    # REVENUE TREND
    html.Div([
        dcc.Graph(id="revenue_trend")
    ], style={
        "backgroundColor": "white",
        "padding": "15px",
        "borderRadius": "12px",
        "boxShadow": "0 4px 8px rgba(0,0,0,0.08)",
        "marginBottom": "25px"
    }),

    # NET PROFIT COMPARISON
    html.Div([
        dcc.Graph(id="profit_comparison")
    ], style={
        "backgroundColor": "white",
        "padding": "15px",
        "borderRadius": "12px",
        "boxShadow": "0 4px 8px rgba(0,0,0,0.08)",
        "marginBottom": "25px"
    }),

    # SHARE PRICE TREND
    html.Div([
        dcc.Graph(id="share_price")
    ], style={
        "backgroundColor": "white",
        "padding": "15px",
        "borderRadius": "12px",
        "boxShadow": "0 4px 8px rgba(0,0,0,0.08)",
        "marginBottom": "25px"
    }),

    # CORRELATION HEATMAP
    html.Div([
        dcc.Graph(id="correlation_heatmap")
    ], style={
        "backgroundColor": "white",
        "padding": "15px",
        "borderRadius": "12px",
        "boxShadow": "0 4px 8px rgba(0,0,0,0.08)",
        "marginBottom": "25px"
    }),

    # STOCK DATA TABLE
    html.Div([

        html.H3(
            "Stock Data Table",
            style={
                "color": "#1F2D3D",
                "marginBottom": "10px"
            }
        ),

        dash_table.DataTable(
            id="table",
            page_size=10,
            style_table={
                "height": "400px",
                "overflowY": "auto"
            },
            style_header={
                "backgroundColor": "#2F4F4F",
                "color": "white",
                "fontWeight": "bold",
                "textAlign": "center"
            },
            style_cell={
                "textAlign": "center",
                "padding": "8px",
                "fontFamily": "Arial",
                "fontSize": "13px"
            },
            style_data={
                "backgroundColor": "#FAFAFA",
                "color": "#333333"
            }
        )

    ], style={
        "backgroundColor": "white",
        "padding": "20px",
        "borderRadius": "12px",
        "boxShadow": "0 4px 8px rgba(0,0,0,0.08)"
    })

], style={
    "backgroundColor": "#EEF2F5",
    "padding": "30px",
    "fontFamily": "Arial"
})

# CALLBACK
@app.callback(
    Output("revenue_trend", "figure"),
    Output("profit_comparison", "figure"),
    Output("share_price", "figure"),
    Output("correlation_heatmap", "figure"),
    Output("table", "data"),
    Output("table", "columns"),
    Input("company_filter", "value")
)

def update_dashboard(selected_company):

    # FILTER DATA
    if selected_company == "All":
        fin_data = financial_data.copy()
        stock_filtered = stock_data.copy()
    else:
        fin_data = financial_data[financial_data["Company"] == selected_company].copy()
        stock_filtered = stock_data[stock_data["Company"] == selected_company].copy()

    # REVENUE TREND
    fig1 = px.line(
        fin_data,
        x="Year",
        y="Revenue",
        color="Company",
        markers=True,
        color_discrete_map=company_colors,
        title="Revenue Trend"
    )

    fig1.for_each_trace(lambda t: t.update(name=company_labels.get(t.name, t.name)))
    fig1.update_traces(line=dict(width=3))

    fig1.update_layout(
        xaxis_title="Year",
        yaxis_title="Revenue in USD Millions",
        plot_bgcolor=graph_background,
        paper_bgcolor="white",
        hovermode="x unified",
        legend_title="Company",
         xaxis=dict(
        tickmode="array",
        tickvals=[2022, 2023, 2024],
        ticktext=["2022", "2023", "2024"]
    ),
        yaxis=dict(range=[15000, 105000])
    )

    # NET PROFIT COMPARISON
    fig2 = px.bar(
        fin_data,
        x="Year",
        y="Net Profit",
        color="Company",
        barmode="group",
        color_discrete_map=company_colors,
        title="Net Profit Comparison"
    )

    fig2.for_each_trace(lambda t: t.update(name=company_labels.get(t.name, t.name)))

    fig2.update_layout(
    xaxis_title="Year",
    yaxis_title="Net Profit in USD Millions",
    plot_bgcolor=graph_background,
    paper_bgcolor="white",
    hovermode="x unified",
    legend_title="Company",
    xaxis=dict(
        tickmode="array",
        tickvals=[2022, 2023, 2024],
        ticktext=["2022", "2023", "2024"]
    ),
    yaxis=dict(
        tickvals=[0, 5000, 10000, 15000, 20000, 25000, 30000, 35000],
        ticktext=["0", "5k", "10k", "15k", "20k", "25k", "30k", "35k"],
        range=[0, 40000]
    )
)

    # SHARE PRICE TREND
    fig3 = px.line(
        stock_filtered,
        x="Date",
        y="Close",
        color="Company",
        color_discrete_map=company_colors,
        title="Share Price Trend"
    )

    fig3.for_each_trace(lambda t: t.update(name=company_labels.get(t.name, t.name)))
    fig3.update_traces(line=dict(width=2.5))

    fig3.update_layout(
        xaxis_title="Date",
        yaxis_title="Share Price (USD)",
        plot_bgcolor=graph_background,
        paper_bgcolor="white",
        hovermode="x unified",
        legend_title="Company"
    )

    # CORRELATION HEATMAP
    if stock_filtered["Company"].nunique() > 1:
        returns_pivot = stock_filtered.pivot(
            index="Date",
            columns="Company",
            values="Return"
        )

        corr_matrix = returns_pivot.corr()

    else:
        corr_matrix = pd.DataFrame(
            [[1]],
            index=stock_filtered["Company"].unique(),
            columns=stock_filtered["Company"].unique()
        )

    fig4 = px.imshow(
        corr_matrix,
        text_auto=".2f",
        color_continuous_scale=[
            (0, "#F7FBFF"),
            (0.5, "#6BAED6"),
            (1, "#08306B")
        ],
        title="Correlation Between Pharmaceutical Stock Returns"
    )

    fig4.update_xaxes(
        tickvals=list(company_labels.keys()),
        ticktext=list(company_labels.values())
    )

    fig4.update_yaxes(
        tickvals=list(company_labels.keys()),
        ticktext=list(company_labels.values())
    )

    fig4.update_layout(
        plot_bgcolor=graph_background,
        paper_bgcolor="white"
    )

    stock_table = stock_filtered.copy()

    stock_table["Company"] = stock_table["Company"].map(company_labels)

    stock_table = stock_table[
        ["Date", "Company", "Close", "Month", "Return", "Return (%)", "MA50", "MA200"]
    ]

    table_data = stock_table.round(4).to_dict("records")
    table_columns = [{"name": i, "id": i} for i in stock_table.columns]

    return (
        fig1,
        fig2,
        fig3,
        fig4,
        table_data,
        table_columns
    )
app.run(jupyter_mode="external", port=8066)

[*********************100%***********************]  3 of 3 completed


Dash app running on http://127.0.0.1:8066/
